Middleware

Middleware provides a way to more tightly  control what happens inside the agent. Middleware is so useful for the following:

1. Tracking agent behaviour with logging, analytics and debugging.
2. Transforming prompts, tool selection, and output formatting.
3. Adding retries, fallback and early termination of logic.
4. Applying ratelimits, guardrails and PII detection.


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

Summarization middleware


Automatically summerizes the conversation history wehn approaching token limits, preserving recent messages while compressing older context.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver



agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)


### Run with thread id

config={"configurable": {
    "thread_id":"test-1"
}}

questions =[
    "what is the capital; of india?","what is 2+4?","what is the capital; of india?","what is 2*10?","what is 2-2?","what is the name of pm of india?"
]


for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool

def search_hotels(city:str)->str:
    """ Search hotels - return long response to use more tokens"""
    return f"""
    Hotels in {city}:
    1. Grand Hotel - 5 star, $350 per night , swimming pool, spa
    2. City Inn - 4 star, $180 per night , business center
    3. Budget stay - 3 star, $75/night, free wifi
    
    """


agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "gpt-4o-mini",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)


config={"configurable": {
    "thread_id":"test-1"
}}

def countTokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4



In [ ]:
cities =["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"find hotels in {city}")]},
        config=config
    )
    tokens = countTokens(response["messages"])
    print(f"{city}:~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")